## Cell 1: Import everything we need

Same libraries as Task 02, plus `time` to measure FPS (frames per second).

In [5]:
from ultralytics import YOLO
import cv2
import time

print("Libraries loaded!")

Libraries loaded!


## Cell 2: Load YOLO model

Same pre-trained model as before. Already knows 80 objects including phones.

In [6]:
model = YOLO('yolov8n.pt')
print("YOLO loaded and ready!")

YOLO loaded and ready!


## Cell 3: Run LIVE detection on webcam

**Before running:**
- Place your phone on the desk
- Make sure webcam can see it

**While running:**
- A window will open showing your webcam with detection boxes
- Move your phone around — watch the box follow it!
- Try hiding the phone — the box disappears
- FPS counter shows in the top-left corner

**To stop:** Press **'q'** on your keyboard while the video window is focused.

### What this code does (line by line):

```
1. Open webcam
2. LOOP:
   a. Capture one frame (photo) from webcam
   b. Send frame to YOLO → "what do you see?"
   c. YOLO returns boxes around detected objects
   d. Draw those boxes on the frame
   e. Show the frame on screen
   f. Check if 'q' was pressed → if yes, stop
   g. Go back to step (a) for next frame
3. Close webcam and window
```

In [7]:
# === OPEN WEBCAM ===
camera = cv2.VideoCapture(0)

if not camera.isOpened():
    print("ERROR: Cannot open webcam!")
    print("Is another app using it? Close that app and retry.")
else:
    print("Webcam opened! Starting detection...")
    print("Press 'q' to quit.")
    print()
    
    frame_count = 0
    detect_count = 0
    start_time = time.time()
    
    # === FPS BOOST SETTINGS ===
    PROCESS_EVERY_N = 3          # Run YOLO every 3rd frame (skip 2, detect 1)
    last_annotated_frame = None   # Store last detection result
    
    # Reduce webcam resolution for speed (640x480 instead of default 1080p)
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    
    # === THE VIDEO LOOP ===
    while True:
        
        # Step A: Capture one frame from webcam
        success, frame = camera.read()
        if not success:
            print("Failed to grab frame. Camera disconnected?")
            break
        
        frame_count += 1
        
        # Step B: Only run YOLO every Nth frame (HUGE speed boost!)
        # Other frames just show the last detection result
        if frame_count % PROCESS_EVERY_N == 0:
            # Run YOLO on this frame
            results = model(frame, verbose=False)
            last_annotated_frame = results[0].plot()
            detect_count += 1
        elif last_annotated_frame is not None:
            # Use previous detection boxes but on current frame
            # This makes video smooth while detection runs less often
            pass
        
        # Step C: Choose what to display
        display_frame = last_annotated_frame if last_annotated_frame is not None else frame
        
        # Step D: Calculate and show FPS
        elapsed = time.time() - start_time
        fps = frame_count / elapsed if elapsed > 0 else 0
        detect_fps = detect_count / elapsed if elapsed > 0 else 0
        
        # Draw FPS counter
        cv2.putText(display_frame, f"Display FPS: {fps:.1f}", (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(display_frame, f"Detection FPS: {detect_fps:.1f}", (10, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 255), 2)
        cv2.putText(display_frame, f"Processing every {PROCESS_EVERY_N}rd frame", (10, 85),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
        
        # Step E: Show the frame in a window
        cv2.imshow('ExamGuard - Phone Detection (Press q to quit)', display_frame)
        
        # Step F: Check if 'q' was pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print(f"\nStopped! {frame_count} frames in {elapsed:.1f} seconds.")
            print(f"Display FPS: {fps:.1f}")
            print(f"Detection FPS: {detect_fps:.1f} (YOLO ran {detect_count} times)")
            break
    
    # === CLEANUP ===
    camera.release()
    cv2.destroyAllWindows()
    print("Webcam released. Done!")

Webcam opened! Starting detection...
Press 'q' to quit.


Stopped! 3027 frames in 107.5 seconds.
Display FPS: 28.2
Detection FPS: 9.4 (YOLO ran 1009 times)
Webcam released. Done!


## Cell 4: What did you observe?

Answer these questions for yourself:

1. **Did YOLO detect your phone?** (Yes/No)
2. **What FPS did you get?** (shown in top-left of video)
   - Above 25 FPS = great, real-time
   - 15-25 FPS = good enough
   - Below 15 FPS = slow, might need smaller model
3. **Did the box flicker?** (appear and disappear on the phone)
   - If yes, the confidence is borderline — we'll fix this in Task 04
4. **Did YOLO detect OTHER things?** (your face, chair, laptop)
   - That's expected! YOLO detects 80 objects.
   - In Task 04, we'll filter to show ONLY phones.

### Understanding FPS

| Your FPS | Meaning | Good enough for ExamGuard? |
|----------|---------|---------------------------|
| 25+ | Real-time, smooth | YES — perfect |
| 15-25 | Slightly choppy but functional | YES — acceptable |
| 5-15 | Noticeable lag | MAYBE — can work but not ideal |
| Below 5 | Very slow | NO — need to optimize |

### If FPS is low:
- We're using YOLOv8**n** (nano = smallest, fastest). This is already the fastest version.
- Low FPS usually means your CPU/GPU is struggling.
- One fix: process every 2nd or 3rd frame instead of every frame. We'll do this in Task 05 if needed.

## Next Step

You now have YOLO running live on your webcam! But it shows boxes around EVERYTHING — people, chairs, laptops.

**Task 04:** Filter detections to show ONLY phones. Ignore everything else.

Open: `../04_phone_only_filter/README.md`